In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import esm 
from tqdm import tqdm


def R2(y_true, y_pred):
    mse = np.mean((y_true - y_pred) ** 2)
    var = np.var(y_true)
    return 1 - mse / var
    

In [4]:
train_df = pd.read_excel("./data/phopt_training.xlsx")
val_df = pd.read_excel("./data/phopt_validation.xlsx")
test_df = pd.read_excel("./data/phopt_testing.xlsx")

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")
train_df.head()

Training set size: 7124
Validation set size: 760
Test set size: 1971


,optimal pH,sequence
0,7.47,MAKFVIRPATAADCSDILRLIKELAKYEYMEEQVILTEKDLLEDGF...
1,6.00,MKRIRSVCIVALTFALIFSGLSLSGQALEKGKSTLVIIHYKEDPNA...
2,8.50,MNRLKSVCAVVLTFALIFSLFPVNSLALEKGKSTLVIIHYKEDKTS...
3,6.50,MLHISRTFAAYLDEMDQIVVLAPKSLGFDGMAPLTLVAPSGEEIPL...
4,7.60,MALSRLSSSSSSSNGSNLFNPFSAAFTLNRPISSDTTATLTIETSL...


In [5]:
train_sequences, train_y = train_df["sequence"].values, train_df["optimal pH"].values
val_sequences, val_y = val_df["sequence"].values, val_df["optimal pH"].values
test_sequences, test_y = test_df["sequence"].values, test_df["optimal pH"].values

#normalize y
mean_y, std_y = np.mean(train_y), np.std(train_y)
train_y = (train_y - mean_y)/std_y
val_y = (val_y - mean_y)/std_y
test_y = (test_y - mean_y)/std_y

In [6]:
batch_size = 4


class ESM2Dataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = ESM2Dataset(train_sequences, train_y)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = ESM2Dataset(val_sequences, val_y)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = ESM2Dataset(test_sequences, test_y)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
#Loading the pre-trained ESM-2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D() #esm.pretrained.esm2_t33_650M_UR50D
batch_converter = alphabet.get_batch_converter()
model.train();
last_layer = len(model.layers)
emb_dim = model.embed_dim

hidden_dim_MLP = 128
lr =  1e-5
num_epochs = 3


batch_labels, batch_strs, batch_tokens = batch_converter([("Protein 1", "KMKMKMKA"),
                                                          ("Protein 2", "MALK")])
batch_labels, batch_strs, batch_tokens

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to C:\Users\athar/.cache\torch\hub\checkpoints\esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to C:\Users\athar/.cache\torch\hub\checkpoints\esm2_t6_8M_UR50D-contact-regression.pt


(['Protein 1', 'Protein 2'],
 ['KMKMKMKA', 'MALK'],
 tensor([[ 0, 15, 20, 15, 20, 15, 20, 15,  5,  2],
         [ 0, 20,  5,  4, 15,  2,  1,  1,  1,  1]]))

In [8]:
class ESM_FFN(torch.nn.Module):
    def __init__(self, esm2_model, emb_dim, hidden_dim, last_layer):
        super(ESM_FFN, self).__init__()
        self.last_layer = last_layer
        self.esm2_model = esm2_model
        self.relu = torch.nn.ReLU()

        self.MLP = torch.nn.Sequential(
            torch.nn.Linear(emb_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, batch_tokens):
        results = self.esm2_model(batch_tokens, repr_layers=[self.last_layer], return_contacts=False)
        cls_repr = results["representations"][self.last_layer][:, 0, :]
        y = self.MLP(cls_repr)
        return y

In [9]:
#Fine-tuning the model
device = torch.device( "cuda" if torch.cuda.is_available() else "cpu")

full_model = ESM_FFN(model, emb_dim = emb_dim, hidden_dim = hidden_dim_MLP, last_layer = last_layer).to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(full_model.parameters(), lr=lr)

In [10]:
for epoch in range(2, num_epochs):
    train_losses = []
    full_model.train()
    #Training loop
    with tqdm(total=len(train_loader), desc=f"Epoch {epoch}") as pbar:
        for batch_seqs, labels in train_loader:
            batch_seq_tuples = [("seq %d" % i, seq) for i, seq in enumerate(batch_seqs)]
            batch_labels, batch_strs, batch_tokens = batch_converter(batch_seq_tuples)

            batch_y_tensor = labels.float().to(device)
            batch_tokens = batch_tokens.to(device)

            optimizer.zero_grad()
            output = full_model(batch_tokens)
            loss = criterion(output.view(-1), batch_y_tensor.view(-1))
            loss.backward()
            optimizer.step()

            # tracking
            train_losses.append(float(loss.item()))
            pbar.set_postfix({"Train Loss": f"{np.mean(train_losses):.6f}"})
            pbar.update(1)

    print("Epoch: %d, Training Loss: %.6f" % (epoch, np.mean(train_losses)))
    
    #Validation loop
    val_true, val_pred = [], []
    full_model.eval()
    for batch_seqs, labels in val_loader:
        batch_seq_tuples = [("seq %d" % i, seq) for i, seq in enumerate(batch_seqs)]
        batch_labels, batch_strs, batch_tokens = batch_converter(batch_seq_tuples)
        batch_y_tensor = labels.float().to(device)

        batch_tokens = batch_tokens.to(device)
        batch_y_tensor = batch_y_tensor.to(device)
        with torch.no_grad():
            output = full_model(batch_tokens)
        val_true.extend(list(batch_y_tensor.cpu().detach().numpy()))
        val_pred.extend(list(output.cpu().detach().numpy().reshape(-1)))
        
    val_true = np.array(val_true).reshape(-1)
    val_pred = np.array(val_pred).reshape(-1)
    mse = np.mean((val_true-val_pred)**2)
    r2 = R2(val_true, val_pred)
    print("Epoch: %d, Validation Loss: %.6f, R2: %.6f" % (epoch, mse, r2))

Epoch 2: 100%|██████████████████████████████████████████████| 1781/1781 [1:03:25<00:00,  2.14s/it, Train Loss=0.776744]


Epoch: 2, Training Loss: 0.776744
Epoch: 2, Validation Loss: 0.729759, R2: 0.314191


In [11]:
#Test loop
test_true, test_pred = [], []
full_model.eval()
for batch_seqs, labels in test_loader:
    batch_seq_tuples = [("seq %d" % i, seq) for i, seq in enumerate(batch_seqs)]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch_seq_tuples)
    batch_y_tensor = labels.float().to(device)

    batch_tokens = batch_tokens.to(device)
    batch_y_tensor = batch_y_tensor.to(device)
    with torch.no_grad():
        output = full_model(batch_tokens)
    test_true.extend(list(batch_y_tensor.cpu().detach().numpy()))
    test_pred.extend(list(output.cpu().detach().numpy().reshape(-1)))
    
test_true = np.array(test_true).reshape(-1)
test_pred = np.array(test_pred).reshape(-1)
mse = np.mean((test_true-test_pred)**2)
r2 = R2(test_true, test_pred)
print("Test Loss: %.6f, R2: %.6f" % (mse, r2))

Test Loss: 0.696006, R2: 0.319510
